In [1]:
import pandas as pd
import numpy as np
import monpa
import time
import multiprocessing
from joblib import Parallel, delayed

# ==========================================
# 定義斷詞 Worker
# ==========================================
def monpa_worker(texts):
    results = []
    for text in texts:
        if pd.isna(text) or str(text).strip() == "":
            results.append("")
            continue
        try:
            words = [item for item in monpa.cut(str(text))]
            words = [w for w in words if len(w.strip()) > 0]
            results.append(" ".join(words))
        except Exception:
            results.append("")
    return results

if __name__ == '__main__':
    start_time = time.time()
    num_cores = multiprocessing.cpu_count()
    
    print(f"🚀 步驟一：Monpa 斷詞預處理 (M4 Pro {num_cores}核全速)")

    # 1. 讀取
    try:
        df_train = pd.read_csv('TrainingData.csv')
        df_test = pd.read_csv('TestingData.csv')
        df_all = pd.concat([df_train, df_test], axis=0, ignore_index=True)
        print(f"   - 資料載入完成: {len(df_all)} 筆")
    except:
        print("❌ 讀取失敗")
        exit()

    # 2. 合併欄位
    df_all['raw_text'] = df_all['Column2'].fillna('') + " " + df_all['Column3'].fillna('')
    raw_texts = df_all['raw_text'].values
    
    # 3. 多核斷詞
    print(f"⚡️ 正在進行斷詞...")
    chunks = np.array_split(raw_texts, num_cores)
    with Parallel(n_jobs=num_cores, backend='loky', verbose=5) as parallel:
        processed_chunks = parallel(delayed(monpa_worker)(chunk) for chunk in chunks)
    
    df_all['text_segmented'] = np.concatenate(processed_chunks)
    
    # 4. 存檔
    output_file = 'Processed_Data.csv'
    # 只存需要的欄位: Label (Column1) 和 斷詞結果 (text_segmented)
    df_all[['Column1', 'text_segmented']].to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"✅ 斷詞完成並已存檔至: {output_file}")
    print(f"⏱️ 總耗時: {(time.time() - start_time)/60:.1f} 分鐘")

+---------------------------------------------------------------------+
  Welcome to MONPA: Multi-Objective NER POS Annotator for Chinese
+---------------------------------------------------------------------+
已找到 model檔。Found model file.
🚀 步驟一：Monpa 斷詞預處理 (M4 Pro 12核全速)
   - 資料載入完成: 47275 筆
⚡️ 正在進行斷詞...


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   4 out of  12 | elapsed:  2.8min remaining:  5.5min
[Parallel(n_jobs=12)]: Done   7 out of  12 | elapsed:  2.8min remaining:  2.0min
[Parallel(n_jobs=12)]: Done  10 out of  12 | elapsed:  2.9min remaining:   34.2s
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:  2.9min finished


✅ 斷詞完成並已存檔至: Processed_Data.csv
⏱️ 總耗時: 2.9 分鐘
